In [ ]:
!pip install transformers datasets librosa torch

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import os
import time

from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio, DatasetDict
from scipy.interpolate import interp1d

In [ ]:
TARGET_IDX = 35

### Φόρτωση finetuned μοντέλου

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model_ft = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft = model_ft.to(device)
model_ft.eval()

In [ ]:
label2id = model_ft.config.label2id
id2label = model_ft.config.id2label

### Φόρτωση dataset (speaker-independent)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()

### Wrapper για forward pass

In [ ]:
class Wrapper(torch.nn.Module):
    """
    Wrapper που επιστρέφει μόνο τα logits του μοντέλου, ώστε forward hooks/
    backward hooks να καταγράψουν activations και gradients στο
    τελευταίο transformer layer χωρίς να χρειάζεται να ξαναγράφεται η κλήση
    του μοντέλου σε κάθε μέθοδο.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x).logits

In [ ]:
wrapped_model = Wrapper(model_ft).to(device).eval()

### Βοηθητικές συναρτήσεις

In [ ]:
def resample_attribution(attr, target_frames=50):
    attr = np.abs(attr)
    x_original = np.linspace(0, 1, len(attr))
    x_target = np.linspace(0, 1, target_frames)
    f = interp1d(x_original, attr, kind='linear')
    resampled = f(x_target)
    resampled = resampled / (resampled.sum() + 1e-10)
    return resampled

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
def load_sample(idx, split="test"):
    sample = dataset[split][idx]
    audio = sample["audio"]
    true_label = sample["emotion"]

    input_values = feature_extractor(
        audio["array"], sampling_rate=feature_extractor.sampling_rate, return_tensors="pt"
    ).input_values.to(device)

    return input_values, true_label, audio["array"]

In [ ]:
input_values, true_label, speech = load_sample(TARGET_IDX, split="test")
print(f"True label: {true_label}")

### Καταγραφή activations/gradients τελευταίου layer

In [ ]:
#Επιλογή target layer (τελευταίο transformer block)
target_layer = model_ft.data2vec_audio.encoder.layers[-1]

acts, grads = {}, {}

def fwd_hook(m, i, o):
    acts['A'] = o[0] if isinstance(o, tuple) else o

def bwd_hook(m, gi, go):
    grads['dA'] = go[0]

h1 = target_layer.register_forward_hook(fwd_hook)
h2 = target_layer.register_full_backward_hook(bwd_hook)

In [ ]:
#Forward + backward
logits = wrapped_model(input_values)
class_idx = logits.argmax(-1).item()
model_ft.zero_grad()
logits[0, class_idx].backward()

A  = acts['A'][0].detach()      # (T, C)
dA = grads['dA'][0].detach()    # (T, C)

def norm(x):
    """Κανονικοποίηση με το μέγιστο (χωρίς αφαίρεση min): ένας σχεδόν
    σταθερός χάρτης παραμένει σταθερός αντί να γίνει τεχνητό σκαλοπάτι."""
    return x / (x.max() + 1e-8)

### Υπολογισμός Grad-CAM, Grad-CAM++, Eigen-CAM, Score-CAM, Ablation-CAM

In [ ]:
#Grad-CAM
w = dA.mean(dim=0)
cam_gradcam = norm(F.relu((w * A).sum(dim=-1))).cpu().numpy()

#Grad-CAM++
g2, g3 = dA**2, dA**3
denom = 2*g2 + A.sum(0, keepdim=True)*g3
denom = torch.where(denom != 0, denom, torch.ones_like(denom))
alpha = g2 / denom
w = (alpha * F.relu(dA)).sum(dim=0)
cam_gradcampp = norm(F.relu((w * A).sum(dim=-1))).cpu().numpy()

#Eigen-CAM
A_np = (A - A.mean(0, keepdim=True)).cpu().numpy()
_, _, Vt = np.linalg.svd(A_np, full_matrices=False)
cam_eigen = np.abs(A_np @ Vt[0])
cam_eigen = cam_eigen / (cam_eigen.max() + 1e-8)

#Score-CAM
T, C = A.shape
up = F.interpolate(A.T.unsqueeze(0), size=input_values.shape[-1], mode='linear')[0]  # (C, samples)
up = (up - up.min(1, keepdim=True).values) / (up.max(1, keepdim=True).values - up.min(1, keepdim=True).values + 1e-8)

scored_weights = []
with torch.no_grad():
    for i in range(0, C, 32):
        masked = up[i:i+32].to(device) * input_values
        out = F.softmax(wrapped_model(masked), dim=-1)[:, class_idx]
        scored_weights.append(out.cpu())
w = torch.cat(scored_weights)
cam_score = norm(F.relu((w.unsqueeze(0) * A.cpu()).sum(-1))).numpy()

#Ablation-CAM
with torch.no_grad():
    base = F.softmax(wrapped_model(input_values), -1)[0, class_idx].item()

abl_weights = torch.zeros(C)
for c in range(C):
    def hook(m, i, o, c=c):
        o2 = (o[0] if isinstance(o, tuple) else o).clone()
        o2[..., c] = 0
        return (o2,) + o[1:] if isinstance(o, tuple) else o2
    hh = target_layer.register_forward_hook(hook)
    with torch.no_grad():
        s = F.softmax(wrapped_model(input_values), -1)[0, class_idx].item()
    hh.remove()
    abl_weights[c] = (base - s) / (base + 1e-8)

cam_ablation = norm(F.relu((abl_weights.unsqueeze(0) * A.cpu()).sum(-1))).numpy()

h1.remove(); h2.remove()

### Οπτικοποίηση και αποθήκευση attributions

In [ ]:
import time

n_samples = input_values.shape[-1]
def upsample(c):
    return F.interpolate(torch.tensor(c).view(1,1,-1), size=n_samples, mode='linear').squeeze().numpy()

cams = {
    "Grad-CAM": cam_gradcam,
    "Grad-CAM++": cam_gradcampp,
    "Eigen-CAM": cam_eigen,
    "Score-CAM": cam_score,
    "Ablation-CAM": cam_ablation,
}

wav = speech
fig, axes = plt.subplots(len(cams)+1, 1, figsize=(11, 1.6*(len(cams)+1)), sharex=True)
axes[0].plot(wav, color="steelblue", linewidth=0.6)
axes[0].set_title("Κυματομορφή εισόδου")

for ax, (name, c) in zip(axes[1:], cams.items()):
    ax.plot(wav, color="lightgray", linewidth=0.5)
    ax.imshow(upsample(c)[np.newaxis, :], cmap="jet", aspect="auto",
          extent=[0, len(wav), wav.min(), wav.max()], alpha=0.6, vmin=0, vmax=1)
    ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=9)

axes[-1].set_xlabel("δείγμα (sample)")
fig.suptitle(f"Πρόβλεψη κλάσης: {model_ft.config.id2label[class_idx]}", fontsize=12)
plt.tight_layout()
plt.show()

for method_name, cam_map in cams.items():
    resampled = resample_attribution(cam_map, target_frames=50)
    filepath = f"{save_dir}/{method_name}_idx{TARGET_IDX}.npy"
    np.save(filepath, resampled)
    print(f"Αποθηκεύτηκε: {method_name}_idx{TARGET_IDX}.npy, shape={resampled.shape}")

#Μικρή καθυστέρηση ώστε να προλάβει να ολοκληρωθεί το sync εγγραφής στο Google Drive πριν τον έλεγχο
time.sleep(3)

for method_name in cams.keys():
    filepath = f"{save_dir}/{method_name}_idx{TARGET_IDX}.npy"
    print(method_name, "->", os.path.exists(filepath), os.path.getsize(filepath) if os.path.exists(filepath) else "N/A")

In [ ]:
from scipy.stats import pearsonr

n_seg = 50
cams50 = {name: resample_attribution(c, n_seg) for name, c in cams.items()}

# 1. Οπτική σύγκριση: είναι πραγματικά "καθρέφτης";
fig, ax = plt.subplots(figsize=(12, 4))
for name in ["Eigen-CAM", "Grad-CAM", "Grad-CAM++", "Score-CAM"]:
    ax.plot(cams50[name], label=name)
ax.set_xlabel("Segment"); ax.set_ylabel("Κανονικοποιημένο attribution")
ax.legend(); plt.show()

In [ ]:
# 2. Συσχέτιση κάθε μεθόδου με την ενέργεια του σήματος (RMS ανά segment)
energy = np.array([np.sqrt((s**2).mean()) for s in np.array_split(speech, n_seg)])
energy /= energy.sum()
for name, c in cams50.items():
    print(f"{name:14s} r(energy) = {pearsonr(c, energy)[0]:+.3f}")

In [ ]:
# 3. Πόση διακύμανση εξηγεί η 1η κύρια συνιστώσα και πώς φαίνεται πριν το abs
A_c = (A - A.mean(0, keepdim=True)).cpu().numpy()
_, S, Vt = np.linalg.svd(A_c, full_matrices=False)
print("Variance explained by PC1:", (S[0]**2) / (S**2).sum())

proj = A_c @ Vt[0]                          # με πρόσημο, πριν το abs
proj50 = np.interp(np.linspace(0, 1, n_seg), np.linspace(0, 1, len(proj)), proj)
print("r(signed proj, Grad-CAM) =", pearsonr(proj50, cams50["Grad-CAM"])[0])
print("r(|proj|,      Grad-CAM) =", pearsonr(np.abs(proj50), cams50["Grad-CAM"])[0])

In [ ]:
# 4. Έλεγχος ευαισθησίας: Eigen-CAM χωρίς centering
_, _, Vt_raw = np.linalg.svd(A.cpu().numpy(), full_matrices=False)
eigen_raw = np.abs(A.cpu().numpy() @ Vt_raw[0])
eigen_raw50 = resample_attribution(eigen_raw, n_seg)
print("r(Eigen centered, Eigen raw) =", pearsonr(cams50["Eigen-CAM"], eigen_raw50)[0])
print("r(Eigen raw, Grad-CAM)       =", pearsonr(eigen_raw50, cams50["Grad-CAM"])[0])

In [ ]:
frame_norm = A.norm(dim=1).cpu().numpy()
print("T =", len(frame_norm))
print("Νόρμα τελευταίων 5 frames:", frame_norm[-5:].round(2))
print("Μέση νόρμα υπόλοιπων:", frame_norm[:-5].mean().round(2))

w_gc = dA.mean(dim=0)
raw_gradcam = (w_gc * A).sum(dim=-1).cpu().numpy()   # πριν από ReLU/min-max
print("Grad-CAM raw, τελευταία 5:", raw_gradcam[-5:].round(3))
print("Grad-CAM raw, median υπόλοιπων:", np.median(raw_gradcam[:-5]).round(3))

# 1. Είναι το gradient ίδιο σε όλα τα frames;
print("max |dA[t]-dA[0]|:", (dA - dA[0]).abs().max().item(),
      "| max |dA|:", dA.abs().max().item())

# 2. Απόσταση κάθε frame από το μέσο frame (αυτό "βλέπει" το PCA)
dev = (A - A.mean(0, keepdim=True)).norm(dim=1).cpu().numpy()
print("Πρώτα 5:", dev[:5].round(2))
print("Τελευταία 5:", dev[-5:].round(2))
print("Μέσος όρος εσωτερικών:", dev[5:-5].mean().round(2))

In [ ]:
edge = 5 * 320    # 5 frames × 320 samples (20 ms στα 16 kHz)
rms = lambda x: np.sqrt((x**2).mean())
print("RMS αρχή:", rms(speech[:edge]), "| τέλος:", rms(speech[-edge:]),
      "| εσωτερικό:", rms(speech[edge:-edge]))